In [1]:
import pandas as pd
from pathlib import Path

data_fix_path = Path('data_fix.csv')
metrix_waktu_path = Path('metrix_waktu.csv')

# Load kedua file dengan delimiter koma
df_data_fix = pd.read_csv(data_fix_path)
df_metrix_waktu = pd.read_csv(metrix_waktu_path)

print('df_data_fix shape      :', df_data_fix.shape)
print('df_metrix_waktu shape  :', df_metrix_waktu.shape)

display(df_data_fix.head(5))
display(df_metrix_waktu.head(5))

df_data_fix shape      : (3012, 73)
df_metrix_waktu shape  : (1867, 3)


,Konten,Judul video,Waktu publikasi video,Durasi,Penayangan tak dilewati,Rata-rata durasi tonton,Persentase penayangan rata-rata (%),Tetap menonton (%),Penonton unik,Penayangan rata-rata per penonton,...,Klik teaser per teaser kartu yang ditampilkan (%),Klik pada elemen layar akhir,Klik per elemen layar akhir yang ditampilkan (%),Elemen layar akhir yang ditampilkan,Penayangan,Waktu tonton (jam),Subscriber,Estimasi pendapatan (IDR),Tayangan,Rasio klik-tayang dari tayangan (%)
0,Total,NaN,NaN,NaN,19920894.0,0:04:01,37.47,80.43,0.0,0.0,...,0.35,36786.0,2.09,1757819.0,20166890.0,1.336489e+06,30937.0,7.368287e+07,126067471.0,13.08
1,KAqnuJ0W8Jo,NEGARA ARAB & MALAYSIA TERDIAM MELIHAT INDONES...,"Aug 23, 2025",740.0,915959.0,0:04:55,39.94,NaN,NaN,NaN,...,NaN,74.0,4.56,1622.0,915959.0,7.520769e+04,5378.0,4.195852e+06,5722428.0,11.54
2,a5VBaiz4yS0,INDONESIA BIKIN ISRAEL NGAMUK‼ SAAT ARAB TAKUT...,"Aug 24, 2025",649.0,319187.0,0:03:59,36.89,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,319187.0,2.122610e+04,790.0,1.052102e+06,1487638.0,14.30
3,PNIbEl5676s,"MALAYSIA KENA ""CERAMAH"" WARGANYA SENDIRI: JANG...","Dec 23, 2025",712.0,286864.0,0:04:56,41.59,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,286864.0,2.359899e+04,448.0,1.005660e+06,1520223.0,15.91
4,jj4tR1MXgGM,GILA KEBERANIAN INDONESIA BIKIN NETANYAHU PANI...,"Sep 26, 2025",752.0,280836.0,0:04:26,35.43,NaN,NaN,NaN,...,NaN,0.0,NaN,0.0,280836.0,2.078276e+04,893.0,8.708020e+05,2217564.0,11.43


,Judul,Tanggal,Jam Upload (WIB)
0,"HINA IQ78! JIRAN SEBUT SURABAYA KOTA BABU, NET...",2026-04-14,20:30
1,KLAIM TUMPUAN DUNIA! WARGA MALAYSIA NGAMUK BBM...,2026-04-14,18:30
2,"GEMPAR! MARKAS IDF JADI ABU, DESTROYER AS DITE...",2026-04-14,16:15
3,"KOCAK! SERING PAMER MINYAK MURAH, KINI WARGA M...",2026-04-14,13:15
4,"KONDISI KRITIS! TRUMP DILARIKAN KE RS, TIM MED...",2026-04-14,10:15


In [3]:
import re

# 1) Normalkan data_fix untuk merge
df_fix = df_data_fix.copy()
df_metrix = df_metrix_waktu.copy()

# Konversi tanggal ke format date
df_fix['Waktu publikasi video'] = pd.to_datetime(df_fix['Waktu publikasi video'], errors='coerce').dt.date
df_metrix['Tanggal'] = pd.to_datetime(df_metrix['Tanggal'], errors='coerce').dt.date

# 2) Buat fungsi ambil 4 kata pertama
def first_n_words(text, n=4):
    if pd.isna(text):
        return ''
    cleaned = str(text).lower()
    cleaned = re.sub(r'[^a-z0-9\s]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return ' '.join(cleaned.split()[:n])

df_fix['judul_4kata'] = df_fix['Judul video'].apply(first_n_words)
df_metrix['judul_4kata'] = df_metrix['Judul'].apply(first_n_words)

# 3) Buat kunci merge: tanggal + 4 kata pertama
df_fix['merge_key'] = df_fix['Waktu publikasi video'].astype(str) + '|' + df_fix['judul_4kata']
df_metrix['merge_key'] = df_metrix['Tanggal'].astype(str) + '|' + df_metrix['judul_4kata']

# 4) Merge: ambil jam upload dari metrix
df_metrix_select = df_metrix[['merge_key', 'Jam Upload (WIB)']].drop_duplicates(subset=['merge_key'], keep='first')
merged_result = df_fix.merge(
    df_metrix_select,
    on='merge_key',
    how='left'
)

# 5) Sort berdasarkan tanggal (ascending) + judul agar terurut rapi
merged_result = merged_result.sort_values(
    by=['Waktu publikasi video', 'Judul video'],
    ascending=[True, True]
).reset_index(drop=True)

# 6) Hapus kolom helper yang tidak diperlukan
merged_result = merged_result.drop(columns=['merge_key', 'judul_4kata'])

print('Jumlah baris data_fix :', len(df_fix))
print('Jumlah baris metrix   :', len(df_metrix))
print('Hasil merge           :', len(merged_result))
print('Kolom jam upload match:', merged_result['Jam Upload (WIB)'].notna().sum())

# 7) Export hasil merge
output_file = 'test_merge.csv'
merged_result.to_csv(output_file, index=False)
print(f'\nBerhasil export: {output_file}')

# Preview hasil
display(merged_result[['Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(20))

Jumlah baris data_fix : 3012
Jumlah baris metrix   : 1867
Hasil merge           : 3012
Kolom jam upload match: 2642

Berhasil export: test_merge.csv


,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,NaN,52756.0
1,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,NaN,20639.0
2,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,NaN,52756.0
3,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,NaN,20639.0
4,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,NaN,69724.0
5,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,NaN,69724.0
6,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,NaN,9630.0
7,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,NaN,3081.0
8,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,NaN,9630.0
9,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,NaN,3081.0


In [5]:
# Analisis pola jam upload dari metrix_waktu untuk mengisi missing values

# 1) Cek pola jam upload yang ada
print('=== ANALISIS POLA JAM UPLOAD ===')
valid_jam = df_metrix['Jam Upload (WIB)'].dropna()
print(f'Total jam upload tersedia: {len(valid_jam)} dari {len(df_metrix)}')
print(f'Missing jam upload: {df_metrix["Jam Upload (WIB)"].isna().sum()}')

print('\nTop 10 jam upload paling sering:')
top_jams = valid_jam.value_counts().head(10)
print(top_jams)

# 2) Ambil mode (jam yang paling sering) sebagai fill value
mode_jam = valid_jam.value_counts().index[0]
mode_freq = valid_jam.value_counts().values[0]
print(f'\nJam upload paling sering: {mode_jam} (muncul {mode_freq} kali)')

# 3) Isi missing values di merged_result menggunakan mode jam
print(f'\n=== FILLING MISSING VALUES ===')
print(f'Sebelum filling: {merged_result["Jam Upload (WIB)"].isna().sum()} missing')

# Gunakan mode sebagai fill value
fill_value = mode_jam
merged_result['Jam Upload (WIB)'] = merged_result['Jam Upload (WIB)'].fillna(fill_value)

print(f'Sesudah filling: {merged_result["Jam Upload (WIB)"].isna().sum()} missing')
print(f'Fill value digunakan: {fill_value}')

# 4) Export hasil dengan data lengkap
output_filled = 'test_merge_filled.csv'
merged_result.to_csv(output_filled, index=False)
print(f'\nBerhasil export: {output_filled}')

# 5) Preview hasil
print('\nPreview data dengan jam upload yang sudah diisi:')
display(merged_result[['Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(30))

=== ANALISIS POLA JAM UPLOAD ===
Total jam upload tersedia: 1867 dari 1867
Missing jam upload: 0

Top 10 jam upload paling sering:
Jam Upload (WIB)
18:30    144
7:30     137
12:30    126
15:30    119
13:15     98
9:30      97
20:30     91
16:15     87
10:15     80
7:15      78
Name: count, dtype: int64

Jam upload paling sering: 18:30 (muncul 144 kali)

=== FILLING MISSING VALUES ===
Sebelum filling: 370 missing
Sesudah filling: 0 missing
Fill value digunakan: 18:30

Berhasil export: test_merge_filled.csv

Preview data dengan jam upload yang sudah diisi:


,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
1,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,20639.0
2,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
3,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,20639.0
4,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,69724.0
5,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,69724.0
6,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
7,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,3081.0
8,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
9,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,3081.0


In [6]:
# Strategi lebih baik: Isi missing jam upload per tanggal
# Setiap tanggal bisa memiliki pola jam upload yang berbeda

print('=== STRATEGI FILLING PER TANGGAL ===')

# 1) Mulai dari merged_result yang belum diisi
result_per_date = merged_result.copy()

# 2) Buat mapping pola jam per tanggal dari metrix
jam_per_date = df_metrix.groupby('Tanggal')['Jam Upload (WIB)'].apply(
    lambda x: x.value_counts().index[0]  # ambil mode (jam paling sering) per tanggal
).to_dict()

print(f'Total tanggal unik dalam metrix: {len(jam_per_date)}')
print(f'\nContoh pola jam per tanggal:')
for tanggal, jam in list(jam_per_date.items())[:10]:
    print(f'  {tanggal}: {jam}')

# 3) Isi missing values berdasarkan tanggal
def fill_jam_by_date(row):
    if pd.isna(row['Jam Upload (WIB)']):
        tanggal = row['Waktu publikasi video']
        # Cari di mapping per tanggal
        if tanggal in jam_per_date:
            return jam_per_date[tanggal]
        else:
            # Fallback ke mode global jika tanggal tidak ada di metrix
            return mode_jam
    else:
        return row['Jam Upload (WIB)']

print(f'\nSebelum filling per tanggal: {result_per_date["Jam Upload (WIB)"].isna().sum()} missing')
result_per_date['Jam Upload (WIB)'] = result_per_date.apply(fill_jam_by_date, axis=1)
print(f'Sesudah filling per tanggal : {result_per_date["Jam Upload (WIB)"].isna().sum()} missing')

# 4) Export hasil
output_per_date = 'test_merge_by_date.csv'
result_per_date.to_csv(output_per_date, index=False)
print(f'\nBerhasil export: {output_per_date}')

# 5) Cek distribusi jam upload sekarang
print(f'\nDistribusi jam upload setelah filling:')
print(result_per_date['Jam Upload (WIB)'].value_counts().head(10))

# 6) Preview hasil
print('\nPreview data dengan jam upload per tanggal:')
display(result_per_date[['Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(30))

=== STRATEGI FILLING PER TANGGAL ===
Total tanggal unik dalam metrix: 343

Contoh pola jam per tanggal:
  2025-05-01: 8:51
  2025-05-02: 9:15
  2025-05-03: 9:15
  2025-05-04: 9:15
  2025-05-05: 9:15
  2025-05-06: 9:30
  2025-05-07: 9:15
  2025-05-08: 9:15
  2025-05-09: 9:15
  2025-05-10: 9:15

Sebelum filling per tanggal: 0 missing
Sesudah filling per tanggal : 0 missing

Berhasil export: test_merge_by_date.csv

Distribusi jam upload setelah filling:
Jam Upload (WIB)
18:30    576
7:30     236
15:30    194
9:30     164
13:15    142
12:30    130
10:15    126
16:15    124
7:15     124
9:15      98
Name: count, dtype: int64

Preview data dengan jam upload per tanggal:


,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
1,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,20639.0
2,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
3,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,20639.0
4,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,69724.0
5,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,69724.0
6,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
7,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,3081.0
8,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
9,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,3081.0


In [7]:
# Perbandingan: reload test_merge.csv asli (dengan 370 missing) lalu isi per tanggal

print('=== PERBANDINGAN STRATEGI ===\n')

# Reload file asli
test_merge_original = pd.read_csv('test_merge.csv')
print(f'test_merge.csv original: {len(test_merge_original)} baris, {test_merge_original["Jam Upload (WIB)"].isna().sum()} missing jam')

# Konversi tanggal untuk match dengan metrix
test_merge_original['Waktu publikasi video'] = pd.to_datetime(
    test_merge_original['Waktu publikasi video'], 
    errors='coerce'
).dt.date

# Isi per tanggal
def fill_jam_by_date_v2(row):
    if pd.isna(row['Jam Upload (WIB)']):
        tanggal = row['Waktu publikasi video']
        if tanggal in jam_per_date:
            return jam_per_date[tanggal]
        else:
            return mode_jam  # fallback
    else:
        return row['Jam Upload (WIB)']

print(f'\nSebelum filling: {test_merge_original["Jam Upload (WIB)"].isna().sum()} missing')
test_merge_original['Jam Upload (WIB)'] = test_merge_original.apply(fill_jam_by_date_v2, axis=1)
print(f'Sesudah filling: {test_merge_original["Jam Upload (WIB)"].isna().sum()} missing')

# Export final
output_final = 'test_merge_final.csv'
test_merge_original.to_csv(output_final, index=False)
print(f'\nBerhasil export final: {output_final}')

# Ringkasan perbandingan
print('\n' + '='*60)
print('RINGKASAN 3 VERSI FILE:')
print('='*60)
print(f'1. test_merge.csv')
print(f'   - Missing: 370 baris')
print(f'   - Strategi: raw dari merge')
print()
print(f'2. test_merge_filled.csv')
print(f'   - Missing: 0 baris')
print(f'   - Strategi: isi semua dengan mode global (18:30)')
print()
print(f'3. test_merge_final.csv')
print(f'   - Missing: 0 baris')
print(f'   - Strategi: isi per tanggal (setiap tanggal pakai jam paling sering di tanggal itu)')
print()
print('Rekomendasi: Gunakan test_merge_final.csv (lebih akurat per tanggal)')
print('='*60)

# Preview final
print('\nPreview test_merge_final.csv:')
display(test_merge_original[['Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(40))

=== PERBANDINGAN STRATEGI ===

test_merge.csv original: 3012 baris, 370 missing jam

Sebelum filling: 370 missing
Sesudah filling: 0 missing

Berhasil export final: test_merge_final.csv

RINGKASAN 3 VERSI FILE:
1. test_merge.csv
   - Missing: 370 baris
   - Strategi: raw dari merge

2. test_merge_filled.csv
   - Missing: 0 baris
   - Strategi: isi semua dengan mode global (18:30)

3. test_merge_final.csv
   - Missing: 0 baris
   - Strategi: isi per tanggal (setiap tanggal pakai jam paling sering di tanggal itu)

Rekomendasi: Gunakan test_merge_final.csv (lebih akurat per tanggal)

Preview test_merge_final.csv:


,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
1,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,20639.0
2,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
3,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,20639.0
4,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,69724.0
5,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,69724.0
6,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
7,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,3081.0
8,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
9,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,3081.0


In [8]:
# Analisis data duplikat di test_merge_final.csv

print('=== ANALISIS DATA DUPLIKAT ===\n')

# Reload final data untuk fresh analysis
df_final = pd.read_csv('test_merge_final.csv')
print(f'Total baris: {len(df_final)}')

# 1) Cek duplikat berdasarkan Konten (Video ID)
dup_konten = df_final.duplicated(subset=['Konten'], keep=False).sum()
print(f'\nDuplikat berdasarkan Konten (Video ID): {dup_konten} baris')
if dup_konten > 0:
    dup_konten_groups = df_final[df_final.duplicated(subset=['Konten'], keep=False)].groupby('Konten').size()
    print(f'  Grup duplikat: {len(dup_konten_groups)}')
    print(f'  Contoh (5 grup teratas):')
    for konten, count in dup_konten_groups.head(5).items():
        print(f'    {konten}: {count} baris')

# 2) Cek duplikat berdasarkan Judul video + Tanggal
dup_judul_tanggal = df_final.duplicated(
    subset=['Judul video', 'Waktu publikasi video'], 
    keep=False
).sum()
print(f'\nDuplikat berdasarkan Judul + Tanggal: {dup_judul_tanggal} baris')
if dup_judul_tanggal > 0:
    dup_jt_groups = df_final[
        df_final.duplicated(subset=['Judul video', 'Waktu publikasi video'], keep=False)
    ].groupby(['Judul video', 'Waktu publikasi video']).size()
    print(f'  Kombinasi unik duplikat: {len(dup_jt_groups)}')
    print(f'  Contoh (5 kombinasi teratas):')
    for (judul, tanggal), count in dup_jt_groups.head(5).items():
        print(f'    {tanggal} - {judul[:50]}...: {count} baris')

# 3) Cek duplikat berdasarkan semua kolom (exact match)
dup_exact = df_final.duplicated(keep=False).sum()
print(f'\nDuplikat exact match (semua kolom sama): {dup_exact} baris')

# 4) Ringkasan
print(f'\n' + '='*60)
print('RINGKASAN DUPLIKAT:')
print('='*60)
print(f'Konten duplikat              : {dup_konten:6d} baris ({dup_konten/len(df_final)*100:.2f}%)')
print(f'Judul + Tanggal duplikat     : {dup_judul_tanggal:6d} baris ({dup_judul_tanggal/len(df_final)*100:.2f}%)')
print(f'Exact match duplikat         : {dup_exact:6d} baris ({dup_exact/len(df_final)*100:.2f}%)')
print(f'Total baris unik            : {len(df_final.drop_duplicates()):6d} baris')
print('='*60)

# 5) Simpan duplikat untuk analisis lebih lanjut
df_duplikat_konten = df_final[df_final.duplicated(subset=['Konten'], keep=False)].sort_values('Konten')
output_duplikat = 'duplikat_konten.csv'
df_duplikat_konten.to_csv(output_duplikat, index=False)
print(f'\nBerhasil export duplikat: {output_duplikat} ({len(df_duplikat_konten)} baris)')

# 6) Preview duplikat
print('\nPreview duplikat berdasarkan Konten:')
display(df_duplikat_konten[['Konten', 'Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(30))

=== ANALISIS DATA DUPLIKAT ===

Total baris: 3012

Duplikat berdasarkan Konten (Video ID): 3012 baris
  Grup duplikat: 1179
  Contoh (5 grup teratas):
     -A6Ikz58Wbg: 2 baris
     -AH9sFHt0-E: 2 baris
     -F6xRD6tGUo: 4 baris
     -JeG6ZKkFJ4: 2 baris
     -JuzZeTJZv8: 2 baris

Duplikat berdasarkan Judul + Tanggal: 3012 baris
  Kombinasi unik duplikat: 1172
  Contoh (5 kombinasi teratas):
    2026-03-15 - 2 NEGARA PALING ANGKUH ASEAN RESMI RUNTUH TOTAL! S...: 2 baris
    2026-03-05 - 20 NEGARA MERINDING! TNI AL KIRIM FREGAT SILUMAN K...: 2 baris
    2026-03-07 - 22 NEGARA SERENTAK DEKLARASIKAN PERANG‼️ MALAYSIA ...: 2 baris
    2025-12-05 - 3 WILAYAH INI INGIN MERDEKA DARI INDONESIA, NO 2 B...: 6 baris
    2026-04-04 - 3500 PASUKAN TENTARA AS MUSNAH SERENTAK! IRAN SUKS...: 2 baris

Duplikat exact match (semua kolom sama): 3012 baris

RINGKASAN DUPLIKAT:
Konten duplikat              :   3012 baris (100.00%)
Judul + Tanggal duplikat     :   3012 baris (100.00%)
Exact match duplikat   

,Konten,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
2972,-A6Ikz58Wbg,5 RIBU MAYAT AS TERGELETAK! 10 JUTA WARGA IRAN...,2026-04-11,13:15,3030.0
2973,-A6Ikz58Wbg,5 RIBU MAYAT AS TERGELETAK! 10 JUTA WARGA IRAN...,2026-04-11,13:15,3030.0
2219,-AH9sFHt0-E,ALARM BAHAYA BAGI KAPAL ASING! RUDAL CM-302 TN...,2026-01-07,18:15,4008.0
2218,-AH9sFHt0-E,ALARM BAHAYA BAGI KAPAL ASING! RUDAL CM-302 TN...,2026-01-07,18:15,4008.0
758,-F6xRD6tGUo,"FILIPINA HISTERIS‼️AKSI GILA SERKA EDI, PRAJUR...",2025-07-30,7:30,197756.0
756,-F6xRD6tGUo,"FILIPINA HISTERIS‼️AKSI GILA SERKA EDI, PRAJUR...",2025-07-30,7:30,30842.0
757,-F6xRD6tGUo,"FILIPINA HISTERIS‼️AKSI GILA SERKA EDI, PRAJUR...",2025-07-30,7:30,30842.0
759,-F6xRD6tGUo,"FILIPINA HISTERIS‼️AKSI GILA SERKA EDI, PRAJUR...",2025-07-30,7:30,197756.0
1086,-JeG6ZKkFJ4,MALAYSIA HEBOH‼ PRABOWO TAK MASUK KORAN JEPANG...,2025-09-07,5:30,8797.0
1087,-JeG6ZKkFJ4,MALAYSIA HEBOH‼ PRABOWO TAK MASUK KORAN JEPANG...,2025-09-07,5:30,8797.0


In [10]:
# Breakdown detail: berapa baris di masing-masing grup duplikat

print('=== DETAIL BREAKDOWN DUPLIKAT ===\n')

# Hitung jumlah duplikat per Konten
dup_per_konten = df_final.groupby('Konten').size().value_counts().sort_index()
print('Distribusi duplikat per Konten (Video ID):')
print(dup_per_konten)
print()

for count, freq in dup_per_konten.items():
    pct = freq / len(df_final.drop_duplicates('Konten')) * 100
    print(f'  {count:2d} copy masing-masing: {freq:4d} video ({pct:6.2f}%)')

# Buat versi clean: hanya ambil 1 baris per Konten
df_clean = df_final.drop_duplicates(subset=['Konten'], keep='first')
print(f'\n' + '='*60)
print(f'Baris original       : {len(df_final):5d}')
print(f'Baris setelah remove : {len(df_clean):5d}')
print(f'Duplikat dihapus     : {len(df_final) - len(df_clean):5d}')
print('='*60)

# Export clean version
output_clean = 'test_merge_final_clean.csv'
df_clean.to_csv(output_clean, index=False)
print(f'\nBerhasil export clean: {output_clean}')

# Preview
print('\nPreview data clean (tanpa duplikat):')
display(df_clean[['Konten', 'Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(30))

=== DETAIL BREAKDOWN DUPLIKAT ===

Distribusi duplikat per Konten (Video ID):
2    923
4    185
6     71
Name: count, dtype: int64

   2 copy masing-masing:  923 video ( 78.29%)
   4 copy masing-masing:  185 video ( 15.69%)
   6 copy masing-masing:   71 video (  6.02%)

Baris original       :  3012
Baris setelah remove :  1179
Duplikat dihapus     :  1833

Berhasil export clean: test_merge_final_clean.csv

Preview data clean (tanpa duplikat):


,Konten,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,OSDYAzU5v-I,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
6,8BIazCUZiU4,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
10,ed73AFKZIio,Harga Keseimbangan Pasar / Ekuilibrium Pasar -...,2020-10-15,18:30,10658.0
14,72Nf9WUc1Kk,"[STRATIFIKASI SOSIAL #1] : Definisi, Kriteria,...",2021-02-06,18:30,4363.0
16,PsqlZW8gysU,"AUSTRALIA & INGGRIS PERINGATI HUT OPM, PAPUA S...",2024-12-01,18:30,16938.0
20,93mbgbFZWf4,PAPUA MASUK DALAM DAFTAR NEGARA-NEGARA BARU YA...,2024-12-04,18:30,11987.0
24,G2dq2a5uysY,"PRABOWO SKAKMAT AUSTRALIA PENYOKONG OPM PAPUA,...",2024-12-11,18:30,13659.0
28,8g3EO5gef3A,DEBAT SENGIT DR CONNIE RAHAKUNDINI BAKRIE VS ...,2024-12-23,18:30,14407.0
30,mhacml69q2E,TIDAK PERLU NEGOSIASI DENGAN PENJAHAT NEGARA S...,2024-12-26,18:30,28199.0
34,kIoPOv7Isz8,DR CONNIE RAHAKUNDINI : INDONESIA HARUS DEKATI...,2024-12-27,18:30,34289.0


In [11]:
# Hitung jumlah duplikat per Konten
dup_per_konten = df_final.groupby('Konten').size().value_counts().sort_index()
print('Distribusi duplikat per Konten (Video ID):')
print(dup_per_konten)
print()

Distribusi duplikat per Konten (Video ID):
2    923
4    185
6     71
Name: count, dtype: int64



In [12]:
# Tampilkan data apa saja yang duplikat di test_merge_final.csv

# Ambil data yang duplikat berdasarkan Konten (Video ID)
duplikat_detail = df_final[
    df_final.duplicated(subset=['Konten'], keep=False)
].sort_values(['Konten', 'Waktu publikasi video', 'Judul video']).reset_index(drop=True)

# Ringkasan: satu baris per video yang duplikat
duplikat_ringkas = (
    duplikat_detail
    .groupby('Konten', as_index=False)
    .agg(
        jumlah_duplikat=('Konten', 'size'),
        judul_contoh=('Judul video', 'first'),
        tanggal=('Waktu publikasi video', 'first')
    )
    .sort_values(['jumlah_duplikat', 'Konten'], ascending=[False, True])
    .reset_index(drop=True)
)

print('Jumlah baris duplikat :', len(duplikat_detail))
print('Jumlah video duplikat :', len(duplikat_ringkas))

print('\nRingkasan video yang duplikat:')
display(duplikat_ringkas)

print('\nDetail baris yang duplikat:')
display(duplikat_detail[['Konten', 'Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']])

Jumlah baris duplikat : 3012
Jumlah video duplikat : 1179

Ringkasan video yang duplikat:


,Konten,jumlah_duplikat,judul_contoh,tanggal
0,-rCZzU_YLq8,6,"MALAYSIA KETAKUTAN, TNI DITERJUNKAN 750 BATALY...",2025-12-03
1,1LRnoOoDP9U,6,PM MALAYSIA MINTA BANTUAN PAMAN PUTIN‼ RUSIA M...,2025-08-08
2,1lFX6oQbHYQ,6,BEGINI BEDANYA‼ LEBIH ENAK MANA⁉ KETURUNAN IND...,2025-11-07
3,2TyE6hNYSF4,6,"PUTIN NGAMUK! SIAPA BERANI SENTUH INDONESIA, S...",2025-05-24
4,3pawgPmemsE,6,INI ALASAN NEGARA ARAB TAK MAU MALAYSIA MASUK ...,2025-11-06
...,...,...,...,...
1174,zgTN3GH9BCg,2,BIKIN HEBOH‼️ THAILAND TAK TAHAN LAGI MILIKI B...,2026-02-04
1175,zhDzWHawsmg,2,PETA DUNIA BERUBAH‼️SINGAPURA DALAM BAHAYA BES...,2025-09-21
1176,zk-5L6ZVxQI,2,"GILA‼ PARADE MILITER DUNIA RUSIA, KORUT, CHINA...",2025-07-22
1177,zwiMD6lW1bY,2,GEMPAR‼️PASPAMPRES TAMPILKAN ALUTSISTA CANGGIH...,2025-08-08



Detail baris yang duplikat:


,Konten,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,-A6Ikz58Wbg,5 RIBU MAYAT AS TERGELETAK! 10 JUTA WARGA IRAN...,2026-04-11,13:15,3030.0
1,-A6Ikz58Wbg,5 RIBU MAYAT AS TERGELETAK! 10 JUTA WARGA IRAN...,2026-04-11,13:15,3030.0
2,-AH9sFHt0-E,ALARM BAHAYA BAGI KAPAL ASING! RUDAL CM-302 TN...,2026-01-07,18:15,4008.0
3,-AH9sFHt0-E,ALARM BAHAYA BAGI KAPAL ASING! RUDAL CM-302 TN...,2026-01-07,18:15,4008.0
4,-F6xRD6tGUo,"FILIPINA HISTERIS‼️AKSI GILA SERKA EDI, PRAJUR...",2025-07-30,7:30,30842.0
...,...,...,...,...,...
3007,zuR_v1frnTI,KAPOK! POLITISI MALAYSIA SOK PALING DERMAWAN B...,2025-12-20,7:15,104227.0
3008,zwiMD6lW1bY,GEMPAR‼️PASPAMPRES TAMPILKAN ALUTSISTA CANGGIH...,2025-08-08,19:00,64461.0
3009,zwiMD6lW1bY,GEMPAR‼️PASPAMPRES TAMPILKAN ALUTSISTA CANGGIH...,2025-08-08,19:00,64461.0
3010,zx7kzzWEWRQ,"SESUMBAR EJEK INDONESIA, KEADAAN TIMOR LESTE M...",2026-02-17,18:15,7421.0


In [13]:
# Jika ada data duplikat, sisakan satu data saja (berdasarkan Konten / Video ID)

df_nodup = pd.read_csv('test_merge_final.csv')

before_rows = len(df_nodup)
df_nodup = df_nodup.drop_duplicates(subset=['Konten'], keep='first').reset_index(drop=True)
after_rows = len(df_nodup)
removed_rows = before_rows - after_rows

print('Total baris sebelum deduplikasi :', before_rows)
print('Total baris sesudah deduplikasi :', after_rows)
print('Jumlah baris duplikat dihapus   :', removed_rows)

output_nodup = 'test_merge_final_nodup.csv'
df_nodup.to_csv(output_nodup, index=False)
print(f'\nBerhasil export: {output_nodup}')

display(df_nodup[['Konten', 'Judul video', 'Waktu publikasi video', 'Jam Upload (WIB)', 'Penayangan']].head(30))

Total baris sebelum deduplikasi : 3012
Total baris sesudah deduplikasi : 1179
Jumlah baris duplikat dihapus   : 1833

Berhasil export: test_merge_final_nodup.csv


,Konten,Judul video,Waktu publikasi video,Jam Upload (WIB),Penayangan
0,OSDYAzU5v-I,Materi dan Contoh Soal Kurva dan Fungsi Permin...,2020-10-08,18:30,52756.0
1,8BIazCUZiU4,Materi dan Contoh Soal Kurva dan Fungsi Penawa...,2020-10-09,18:30,9630.0
2,ed73AFKZIio,Harga Keseimbangan Pasar / Ekuilibrium Pasar -...,2020-10-15,18:30,10658.0
3,72Nf9WUc1Kk,"[STRATIFIKASI SOSIAL #1] : Definisi, Kriteria,...",2021-02-06,18:30,4363.0
4,PsqlZW8gysU,"AUSTRALIA & INGGRIS PERINGATI HUT OPM, PAPUA S...",2024-12-01,18:30,16938.0
5,93mbgbFZWf4,PAPUA MASUK DALAM DAFTAR NEGARA-NEGARA BARU YA...,2024-12-04,18:30,11987.0
6,G2dq2a5uysY,"PRABOWO SKAKMAT AUSTRALIA PENYOKONG OPM PAPUA,...",2024-12-11,18:30,13659.0
7,8g3EO5gef3A,DEBAT SENGIT DR CONNIE RAHAKUNDINI BAKRIE VS ...,2024-12-23,18:30,14407.0
8,mhacml69q2E,TIDAK PERLU NEGOSIASI DENGAN PENJAHAT NEGARA S...,2024-12-26,18:30,28199.0
9,kIoPOv7Isz8,DR CONNIE RAHAKUNDINI : INDONESIA HARUS DEKATI...,2024-12-27,18:30,34289.0
